<a href="https://colab.research.google.com/github/kjmobile/lb2/blob/main/0-2_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment (0-2): Multiple Regression for Flight Delay Prediction

Do this after notebooks 0, 1 and 2.

**Goal.** Predict a flight's arrival delay from its departure delay and the weather. Weather is a category, so
set **'Clear'** as the baseline (reference) category: every other weather is compared with a clear day
(dummy variables).

1. **Load the data** from the class database with SQL
2. **Check the data**
3. **Build two models**: a simple regression (given) and a multiple regression with the weather (Q1)
4. **Read the results** (Q2 to Q5)
5. **Look at two plots** (Q6)

**Submit:** rename the file `0-2_assignment_yourname.ipynb`, then File → Download → Download .ipynb, and upload
it to Canvas.

In [ ]:
!pip install pymysql

In [ ]:
# Step 0: Import necessary libraries
import pymysql
import pandas as pd
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')  # Hide warning messages for cleaner output

## Step 1: Connect to DB and Load Data
We connect to the database and load data from the flights_delay table which contains all the engineered features we need.

In [ ]:
# Create a direct connection to the database
conn = pymysql.connect(
    host='database-klee.cbgcswckszgl.us-east-1.rds.amazonaws.com',
    user='erau',
    password='1212',
    database='airline_db',
    charset='utf8mb4'
)

# Query data from the flights_delay table
query = """
SELECT
    flight_id,
    departure_delay_mins,
    delay_minutes AS arrival_delay_mins,
    weather_condition,
    day_of_week,
    aircraft_type,
    airline_code
FROM
    flights_delay
WHERE
    departure_delay_mins IS NOT NULL
    AND delay_minutes IS NOT NULL
    AND weather_condition IS NOT NULL;
"""

# Execute the query and load data into a DataFrame
df = pd.read_sql_query(query, conn)

# Close the database connection
conn.close()

In [ ]:
df.head()

## Step 2: Check the Data
Count the missing values in each column.

In [ ]:
df.isnull().sum()

Every column shows **0**: there are no missing values, so there is nothing to clean here.
Real data usually has some. Then you would remove those rows with `df = df.dropna()` before modeling.

## Step 3: Build the Models

**A simple linear regression is one line.** Run it: arrival delay from departure delay only.

In [ ]:
simple = smf.ols('arrival_delay_mins ~ departure_delay_mins', data=df).fit()
print(simple.params)
print('R2:', round(simple.rsquared, 3))

**A multiple regression is the same line with one more input after a `+`.**

Weather is a **category** (Clear, Cloudy, Rain, ...), not a number. Wrap it in `C(...)` and name the baseline:

```python
C(weather_condition, Treatment(reference="Clear"))
```

That is the only new part.

In [ ]:
# Q1 - Build the model using 'Clear' as the reference category (note the capital C)

## Step 4: Read the Results

In [ ]:
# Q2 - Print the regression results summary

Answer in the text cells below, using numbers from your summary.

**Q3.** What does the coefficient of `departure_delay_mins` mean? If a flight leaves one minute later, how many
minutes later does it arrive?

**Your answer:**

**Q4.** Compared with a **Clear** day, which weather adds the most arrival delay, and by how many minutes?

**Your answer:**

**Q5.** What is the R² of your model? Compare it with the simple regression's R² in Step 3. How much did adding the weather help?

**Your answer:**

## Step 5: Look at Two Plots

**Plot 1: actual vs. predicted.** Each dot is one flight. On the red line the prediction equals the actual delay.
The closer the dots sit to the line, the better the model.

In [ ]:
plt.scatter(results.fittedvalues, df['arrival_delay_mins'], s=10)
plt.plot([0, 140], [0, 140], color='red')
plt.xlabel('predicted arrival delay (minutes)')
plt.ylabel('actual arrival delay (minutes)')
plt.show()

**Plot 2: average arrival delay by weather.** These are plain averages, before the model takes departure delay
into account.

In [ ]:
df.groupby('weather_condition')['arrival_delay_mins'].mean().sort_values().plot(kind='barh')
plt.xlabel('average arrival delay (minutes)')
plt.show()

**Q6.** In Plot 2, how much longer is the average delay on a Thunderstorm day than on a Clear day? Is that gap
bigger or smaller than the Thunderstorm coefficient in your model? Why might they differ? (Hint: what else is
different about flights on stormy days?)

**Your answer:**

---
**Takeaway.** Both models were **one line of code**. The only new idea in the multiple regression was telling
Python that weather is a category and which category is the baseline.